# Exercise: a second batch of patients

A second batch of patients from the trial was recorded in `predimed_2.csv`. Use the split-apply-combine skills from the previous exercises to answer the two questions below.

In [ ]:
# import libraries
import pandas as pd

## Load the data

In [ ]:
# load the data

df_patients = pd.read_csv('predimed_2.csv')
df_patients.head()

In [ ]:
df_patients.shape

## Question 1: mean age per diet group


In [ ]:
# the natural attempt would be
df_patients.groupby('group')['age'].mean()
df_patients.groupby('group')['sex_age'].mean()
# but age is not a column

To be able to apply the *groupby* operation, we need tidy data.
Let's create a tidy datatable.

In [ ]:
# solution

df_patients[['sex', 'age']] = (
    df_patients['sex_age']
    .str.rsplit('_', n=1, expand=True)) # n - number of splits; only one here '_'

df_patients['age'] = pd.to_numeric(df_patients['age'], downcast='integer')

In [ ]:
df_patients.groupby('group')['age'].mean()

            ## Question 2: number of cardiovascular events per smoking status

In [ ]:
# the natural attempt
df_patients.groupby('smoke')['event'].sum()
# but smoke is not a column

In [ ]:
# check if we can use *'patient-id'* as id-column
print(len(df_patients))
print(len(df_patients['patient-id'].unique()))
print(len(set(df_patients['patient-id'].unique())))

In [ ]:
# solution 1, create from dummy column by hand
smoke_cols = [
    'smoke_never',
    'smoke_former',
    'smoke_current'
]

df_patients_tidy = df_patients.copy()
df_patients_tidy['smoke_status_from_dummy'] = (
    pd.from_dummies(df_patients[smoke_cols], sep='_')
      ['smoke']
)

In [ ]:
df_patients_tidy_2 = (
    df_patients.melt(
        id_vars = df_patients.columns[~df_patients.columns.isin(smoke_cols)],
        value_vars = smoke_cols,
        var_name = 'smoke_status',
        ignore_index = False)
    .query('value == 1')
    .drop(columns='value'))

df_patients_tidy_2['smoke_status'] = (
    df_patients_tidy_2['smoke_status'].str.removeprefix('smoke_')
)

In [ ]:
# make sure the dataframes are sorted identically
df_patients_tidy = df_patients_tidy.sort_values(['patient-id'])
df_patients_tidy_2 = df_patients_tidy_2.sort_values(['patient-id'])

In [ ]:
# check if both algorithm have the same result
df_patients_tidy['smoke_status_from_dummy'].equals(df_patients_tidy_2['smoke_status'])

### answer the question

* number of cardiavascular events per smoking status

In [ ]:
df_patients_tidy_2.groupby('smoke_status')['event'].sum()

In [ ]:
df_patients_tidy.groupby('smoke_status_from_dummy')['event'].sum()